# Finalisimo2000 en Google Colab

Abrí este cuaderno directamente en Colab: [Abrir con Google Colab](https://colab.research.google.com/github/strumiaf5-del/Finalisimo2000/blob/main/finalisimo_colab.ipynb).

Ejecuta las celdas en orden para clonar el proyecto, instalar dependencias, iniciar la API y abrir el frontend. La sesión de Colab usa almacenamiento temporal; al reiniciarse el runtime, vuelve a ejecutar las celdas.

In [ ]:
#@title 1) Clonar el repositorio
from pathlib import Path

repo_path = Path('/content/Finalisimo2000')
if (repo_path / '.git').exists():
    !git -C /content/Finalisimo2000 pull --ff-only
else:
    !git clone --depth 1 https://github.com/strumiaf5-del/Finalisimo2000.git /content/Finalisimo2000

%cd /content/Finalisimo2000
print(f'Repo activo: {Path.cwd()}')

In [ ]:
#@title 2) Instalar dependencias del backend y frontend
import shutil
import subprocess

if shutil.which('ffmpeg') is None:
    !apt-get -qq update
    !apt-get -qq install -y ffmpeg

!python -m pip install -q -r backend/requirements.txt
!npm --prefix frontend ci

In [ ]:
#@title 3) Iniciar backend y frontend
import os
import subprocess
import sys
import time
from pathlib import Path

import requests

repo_path = Path('/content/Finalisimo2000')
backend_dir = repo_path / 'backend'
frontend_dir = repo_path / 'frontend'

for process_name in ('backend_process', 'frontend_process'):
    previous = globals().get(process_name)
    if previous is not None and previous.poll() is None:
        previous.terminate()

backend_env = os.environ.copy()
backend_env['PYTHONPATH'] = str(backend_dir)
backend_log_file = open('/content/finalisimo-backend.log', 'w')
backend_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=backend_dir,
    env=backend_env,
    stdout=backend_log_file,
    stderr=subprocess.STDOUT,
)

frontend_env = os.environ.copy()
frontend_env['BACKEND_PROXY_TARGET'] = 'http://127.0.0.1:8000'
frontend_log_file = open('/content/finalisimo-frontend.log', 'w')
frontend_process = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173', '--strictPort'],
    cwd=frontend_dir,
    env=frontend_env,
    stdout=frontend_log_file,
    stderr=subprocess.STDOUT,
)

print(f'Backend PID: {backend_process.pid}')
print(f'Frontend PID: {frontend_process.pid}')
print('Logs: /content/finalisimo-backend.log y /content/finalisimo-frontend.log')

In [ ]:
#@title 4) Comprobar API, proxy y abrir la interfaz
import time
from pathlib import Path


def wait_for_http(url, timeout_seconds=300):
    deadline = time.monotonic() + timeout_seconds
    last_error = None
    while time.monotonic() < deadline:
        try:
            return requests.get(url, timeout=10)
        except requests.RequestException as exc:
            last_error = exc
            time.sleep(2)
    raise TimeoutError(f'No respondió {url} en {timeout_seconds}s: {last_error}')


backend_health = wait_for_http('http://127.0.0.1:8000/health')
print('Backend /health:', backend_health.status_code, backend_health.text[:500])
if backend_health.status_code != 200:
    raise RuntimeError('El backend respondió con error. Revisa /content/finalisimo-backend.log')

frontend_response = wait_for_http('http://127.0.0.1:5173/')
proxy_health = requests.get('http://127.0.0.1:5173/api/health', timeout=20)
print('Frontend:', frontend_response.status_code)
print('Proxy API:', proxy_health.status_code)
if frontend_response.status_code != 200 or proxy_health.status_code != 200:
    raise RuntimeError('Falló el frontend o el proxy de API. Revisa los logs en /content/')

from google.colab.output import eval_js
from IPython.display import Markdown, display

frontend_url = eval_js('google.colab.kernel.proxyPort(5173)')
display(Markdown(f'### [Abrir Finalisimo2000]({frontend_url})\n\nLa API está disponible para la interfaz por `/api`; preview y métricas en vivo pasan por el mismo origen.'))

## Pruebas

Abrí el enlace generado en la celda anterior, inicia sesión o crea un usuario desde la interfaz y carga un archivo de audio. Desde el estudio puedes probar preview en vivo, medidores/GR y mastering. El frontend reenvía las llamadas `/api/...` al backend del runtime de Colab; no hace falta editar la URL de API ni habilitar CORS manualmente.

El runtime de Colab es efímero y puede desconectarse por inactividad. Si reinicias la sesión, vuelve a ejecutar las celdas desde el principio.

In [ ]:
#@title Detener servicios (opcional)
for process in (globals().get('frontend_process'), globals().get('backend_process')):
    if process is not None and process.poll() is None:
        process.terminate()
        print(f'Servicio detenido (PID {process.pid})')